# 03 — PCA: Customer Behaviour Structure

**Business question:** Can a small number of latent dimensions summarize customer value, tenure and service intensity?

PCA is an exploratory layer, not the main decision model.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from src.features import add_business_features

path = ROOT/"data/processed/demo_customers.csv"
if not path.exists():
    exec((ROOT/"data/generate_demo_data.py").read_text())
    main()
df = add_business_features(pd.read_csv(path))

In [ ]:
cols = [c for c in ["tenure","monthly_charges","total_charges","number_of_services"] if c in df]
X = df[cols].fillna(df[cols].median())
Xs = StandardScaler().fit_transform(X)

pca = PCA()
Z = pca.fit_transform(Xs)

pd.DataFrame({
    "component": [f"PC{i+1}" for i in range(len(cols))],
    "explained_variance_ratio": pca.explained_variance_ratio_
})

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=cols,
    columns=[f"PC{i+1}" for i in range(len(cols))]
)
loadings

In [ ]:
plt.scatter(Z[:,0], Z[:,1], c=df["churn_flag"], alpha=0.35)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Customers in PCA space (colored by churn)")
plt.tight_layout()
plt.show()

Interpret PCs in business language: e.g. *customer maturity/value* or *service intensity*.
Avoid claiming causal meaning from PCA.